# Phase 3: Model Training and Churn Evaluation
## Customer Churn & LTV Prediction Engine

This notebook trains multiple machine learning classifiers to predict customer churn probability. It evaluates them using metrics suitable for imbalanced datasets (ROC-AUC, Precision-Recall, F1-Score) and serializes the champion model.

### Objectives:
1. Load preprocessed features and labels.
2. Train baseline models (Logistic Regression, Random Forest, XGBoost).
3. Fine-tune hyperparameters using GridSearchCV/RandomizedSearchCV.
4. Assess model quality using confusion matrices, ROC curves, and calibration curves.
5. Save the best model and parameters to the `models/` directory.

In [1]:
import pandas as pd
import joblib

# 1. Load the split datasets
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')

# We use .values.ravel() to convert the y dataframes into 1D arrays, 
# which prevents Scikit-Learn from showing warnings during training.
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# 2. Load the saved preprocessor object
preprocessor = joblib.load('../models/preprocessor.joblib')

# 3. Transform the raw features into model-ready numeric features
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# 4. Verify shapes
print("Datasets loaded and transformed successfully!")
print(f"X_train_trans shape: {X_train_trans.shape}")
print(f"X_test_trans shape: {X_test_trans.shape}")


Datasets loaded and transformed successfully!
X_train_trans shape: (5634, 38)
X_test_trans shape: (1409, 38)


In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier

# ── Baseline XGBoost Classifier ───────────────────────────────────────────────
print("Training XGBoost Classifier (baseline)...")
xgb_baseline = XGBClassifier(
    n_estimators   = 100,
    max_depth       = 4,
    learning_rate   = 0.1,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    use_label_encoder=False,
    eval_metric     = "logloss",
    random_state    = 42,
    n_jobs          = -1,
)
xgb_baseline.fit(X_train, y_train,
                 eval_set=[(X_test, y_test)],
                 verbose=False)
y_pred_base  = xgb_baseline.predict(X_test)
y_prob_base  = xgb_baseline.predict_proba(X_test)[:, 1]

# ── Metrics (no sklearn) ──────────────────────────────────────────────────────
def accuracy(y_true, y_pred):
    return (y_true == y_pred).mean()

def roc_auc(y_true, y_prob):
    # Trapezoidal AUC computed from scratch
    thresholds = sorted(set(y_prob), reverse=True)
    tpr_list, fpr_list = [0.0], [0.0]
    pos = y_true.sum(); neg = len(y_true) - pos
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        tp = ((pred == 1) & (y_true == 1)).sum()
        fp = ((pred == 1) & (y_true == 0)).sum()
        tpr_list.append(tp / pos if pos > 0 else 0)
        fpr_list.append(fp / neg if neg > 0 else 0)
    tpr_list.append(1.0); fpr_list.append(1.0)
    return float(np.trapz(tpr_list, fpr_list))

acc_base = accuracy(y_test.values, y_pred_base)
auc_base = roc_auc(y_test.values, y_prob_base)
print(f"Baseline XGBoost — Accuracy: {acc_base:.4f}  |  ROC-AUC: {auc_base:.4f}")


In [ ]:
# ── XGBoost Hyperparameter Tuning with xgb.cv() ───────────────────────────────
import xgboost as xgb

# Convert to DMatrix (XGBoost's optimised data format)
dtrain = xgb.DMatrix(X_train, label=y_train)

# Parameter grid — we manually loop (replaces GridSearchCV)
param_grid = [
    {"max_depth": d, "n_estimators": n, "learning_rate": lr,
     "subsample": 0.8, "colsample_bytree": 0.8,
     "eval_metric": "auc", "objective": "binary:logistic",
     "seed": 42}
    for d  in [4, 6]
    for n  in [100, 200]
    for lr in [0.05, 0.1]
]

best_auc, best_params = 0.0, None
print("Running XGBoost cross-validation grid search...")
for params in param_grid:
    n = params.pop("n_estimators")
    cv_result = xgb.cv(
        params, dtrain,
        num_boost_round=n,
        nfold=3,
        metrics="auc",
        early_stopping_rounds=20,
        verbose_eval=False,
    )
    mean_auc = cv_result["test-auc-mean"].iloc[-1]
    if mean_auc > best_auc:
        best_auc = mean_auc
        best_params = {**params, "n_estimators": n}
    params["n_estimators"] = n   # restore for next iter

print(f"Best CV ROC-AUC: {best_auc:.4f}")
print(f"Best params: {best_params}")

# ── Retrain with best params on full training set ─────────────────────────────
n_est = best_params.pop("n_estimators")
best_xgb = XGBClassifier(
    **{k: v for k, v in best_params.items()
       if k not in ("eval_metric", "objective", "seed")},
    n_estimators=n_est,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
best_xgb.fit(X_train, y_train, verbose=False)
y_pred_tuned = best_xgb.predict(X_test)
y_prob_tuned = best_xgb.predict_proba(X_test)[:, 1]

acc_tuned = accuracy(y_test.values, y_pred_tuned)
auc_tuned = roc_auc(y_test.values, y_prob_tuned)
print(f"\nTuned XGBoost — Accuracy: {acc_tuned:.4f}  |  ROC-AUC: {auc_tuned:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Confusion Matrix (numpy, no sklearn) ──────────────────────────────────────
def confusion_matrix_np(y_true, y_pred):
    classes = sorted(set(y_true))
    n = len(classes)
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t)][int(p)] += 1
    return cm

cm = confusion_matrix_np(y_test.values, y_pred_tuned)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Stayed (0)", "Churned (1)"],
            yticklabels=["Stayed (0)", "Churned (1)"])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix — Tuned XGBoost")

# ── ROC Curve (numpy trapz, no sklearn) ──────────────────────────────────────
thresholds = sorted(set(y_prob_tuned), reverse=True)
tpr_pts, fpr_pts = [0.0], [0.0]
pos = y_test.values.sum(); neg = len(y_test) - pos
for t in thresholds:
    pred = (y_prob_tuned >= t).astype(int)
    tp = ((pred == 1) & (y_test.values == 1)).sum()
    fp = ((pred == 1) & (y_test.values == 0)).sum()
    tpr_pts.append(tp / pos); fpr_pts.append(fp / neg)
tpr_pts.append(1.0); fpr_pts.append(1.0)
auc_score = float(np.trapz(tpr_pts, fpr_pts))

axes[1].plot(fpr_pts, tpr_pts, color="darkorange", lw=2,
             label=f"XGBoost (AUC = {auc_score:.4f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Tuned XGBoost")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()
print(f"Final ROC-AUC: {auc_score:.4f}")


### Classification Metrics: Precision, Recall & F1-Score
Full per-class breakdown — important for imbalanced churn datasets.

In [ ]:
# Full Classification Metrics (Precision, Recall, F1) - pure numpy, no sklearn
import numpy as np

def precision_recall_f1(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_0        = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    support_1 = int(y_true.sum())
    support_0 = len(y_true) - support_1

    print("\n--- Classification Report (XGBoost Churn Model) ---")
    print(f"{'':>20} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
    print("-" * 65)
    print(f"{'Retained (0)':>20} {precision_0:>10.4f} {recall_0:>10.4f} {f1_0:>10.4f} {support_0:>10}")
    print(f"{'Churned (1)':>20} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f} {support_1:>10}")
    print("-" * 65)

    macro_p = (precision + precision_0) / 2
    macro_r = (recall + recall_0) / 2
    macro_f = (f1 + f1_0) / 2
    w_p = (precision * support_1 + precision_0 * support_0) / len(y_true)
    w_r = (recall * support_1 + recall_0 * support_0) / len(y_true)
    w_f = (f1 * support_1 + f1_0 * support_0) / len(y_true)
    print(f"{'macro avg':>20} {macro_p:>10.4f} {macro_r:>10.4f} {macro_f:>10.4f} {len(y_true):>10}")
    print(f"{'weighted avg':>20} {w_p:>10.4f} {w_r:>10.4f} {w_f:>10.4f} {len(y_true):>10}")

    return {"precision": precision, "recall": recall, "f1": f1,
            "precision_0": precision_0, "recall_0": recall_0, "f1_0": f1_0}

clf_metrics = precision_recall_f1(y_test.values, y_pred_tuned)


### Classification Metrics: Precision, Recall & F1-Score
Full per-class breakdown — important for imbalanced churn datasets.

In [ ]:
# Full Classification Metrics (Precision, Recall, F1) - pure numpy, no sklearn
import numpy as np

def precision_recall_f1(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_0        = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    support_1 = int(y_true.sum())
    support_0 = len(y_true) - support_1

    print("\n--- Classification Report (XGBoost Churn Model) ---")
    print(f"{'':>20} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
    print("-" * 65)
    print(f"{'Retained (0)':>20} {precision_0:>10.4f} {recall_0:>10.4f} {f1_0:>10.4f} {support_0:>10}")
    print(f"{'Churned (1)':>20} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f} {support_1:>10}")
    print("-" * 65)

    macro_p = (precision + precision_0) / 2
    macro_r = (recall + recall_0) / 2
    macro_f = (f1 + f1_0) / 2
    w_p = (precision * support_1 + precision_0 * support_0) / len(y_true)
    w_r = (recall * support_1 + recall_0 * support_0) / len(y_true)
    w_f = (f1 * support_1 + f1_0 * support_0) / len(y_true)
    print(f"{'macro avg':>20} {macro_p:>10.4f} {macro_r:>10.4f} {macro_f:>10.4f} {len(y_true):>10}")
    print(f"{'weighted avg':>20} {w_p:>10.4f} {w_r:>10.4f} {w_f:>10.4f} {len(y_true):>10}")

    return {"precision": precision, "recall": recall, "f1": f1,
            "precision_0": precision_0, "recall_0": recall_0, "f1_0": f1_0}

clf_metrics = precision_recall_f1(y_test.values, y_pred_tuned)


### Classification Metrics: Precision, Recall & F1-Score
Full per-class breakdown — important for imbalanced churn datasets.

In [ ]:
# Full Classification Metrics (Precision, Recall, F1) - pure numpy, no sklearn
import numpy as np

def precision_recall_f1(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_0        = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    support_1 = int(y_true.sum())
    support_0 = len(y_true) - support_1

    print("\n--- Classification Report (XGBoost Churn Model) ---")
    print(f"{'':>20} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
    print("-" * 65)
    print(f"{'Retained (0)':>20} {precision_0:>10.4f} {recall_0:>10.4f} {f1_0:>10.4f} {support_0:>10}")
    print(f"{'Churned (1)':>20} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f} {support_1:>10}")
    print("-" * 65)

    macro_p = (precision + precision_0) / 2
    macro_r = (recall + recall_0) / 2
    macro_f = (f1 + f1_0) / 2
    w_p = (precision * support_1 + precision_0 * support_0) / len(y_true)
    w_r = (recall * support_1 + recall_0 * support_0) / len(y_true)
    w_f = (f1 * support_1 + f1_0 * support_0) / len(y_true)
    print(f"{'macro avg':>20} {macro_p:>10.4f} {macro_r:>10.4f} {macro_f:>10.4f} {len(y_true):>10}")
    print(f"{'weighted avg':>20} {w_p:>10.4f} {w_r:>10.4f} {w_f:>10.4f} {len(y_true):>10}")

    return {"precision": precision, "recall": recall, "f1": f1,
            "precision_0": precision_0, "recall_0": recall_0, "f1_0": f1_0}

clf_metrics = precision_recall_f1(y_test.values, y_pred_tuned)


In [ ]:
import os

os.makedirs("../models", exist_ok=True)

# Save XGBoost model in native JSON format
model_path = "../models/churn_model.json"
best_xgb.save_model(model_path)

print(f"Champion XGBoost model saved to: {os.path.abspath(model_path)}")
print(f"Final Accuracy : {acc_tuned:.4f}")
print(f"Final ROC-AUC  : {auc_tuned:.4f}")

# ── Feature Importance ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd

feat_imp = pd.Series(
    best_xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("Top-15 Feature Importances — XGBoost Churn Model")
ax.set_xlabel("Importance Score (gain)")
plt.tight_layout()
plt.show()


### SHAP Explainability
SHAP (SHapley Additive exPlanations) quantifies each feature's contribution to individual predictions — essential for business stakeholder communication.

In [ ]:
# SHAP Values - Explainability for Business Stakeholders
import shap
import matplotlib.pyplot as plt
import pandas as pd

print("Computing SHAP values for the tuned XGBoost model...")

# TreeExplainer is the fastest and most accurate for XGBoost
explainer = shap.TreeExplainer(best_xgb)

# Use a sample for speed (SHAP on full test set can be slow)
X_shap = X_test.sample(min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values computed for {len(X_shap)} samples x {X_shap.shape[1]} features")

# --- 1. Beeswarm summary plot ---
fig1, ax1 = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, plot_type="dot",
                  max_display=15, show=False)
plt.title("SHAP Feature Impact (Beeswarm) - Churn Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_beeswarm.png")

# --- 2. Bar chart - mean |SHAP| per feature ---
mean_shap = pd.Series(
    abs(shap_values).mean(axis=0),
    index=X_shap.columns
).sort_values(ascending=False).head(15)

fig2, ax2 = plt.subplots(figsize=(10, 6))
mean_shap.plot(kind="barh", ax=ax2, color="steelblue")
ax2.invert_yaxis()
ax2.set_xlabel("Mean |SHAP Value| (average impact on model output)")
ax2.set_title("Top-15 Features by SHAP Importance - Churn Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_feature_importance.png")

# --- 3. Waterfall plot for a single high-risk customer ---
# Pick the customer most likely to churn
high_risk_idx = y_prob_tuned.argmax()
shap_single = explainer(X_shap.iloc[[high_risk_idx]])

fig3, ax3 = plt.subplots(figsize=(10, 6))
shap.plots.waterfall(shap_single[0], max_display=12, show=False)
plt.title(f"SHAP Waterfall - Highest Churn Risk Customer (prob={y_prob_tuned[high_risk_idx]:.2%})",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_waterfall_highrisk.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_waterfall_highrisk.png")
print("\nKey SHAP Insights:")
for feat, val in mean_shap.head(5).items():
    print(f"  {feat}: {val:.4f} avg |SHAP|")


### SHAP Explainability
SHAP (SHapley Additive exPlanations) quantifies each feature's contribution to individual predictions — essential for business stakeholder communication.

In [ ]:
# SHAP Values - Explainability for Business Stakeholders
import shap
import matplotlib.pyplot as plt
import pandas as pd

print("Computing SHAP values for the tuned XGBoost model...")

# TreeExplainer is the fastest and most accurate for XGBoost
explainer = shap.TreeExplainer(best_xgb)

# Use a sample for speed (SHAP on full test set can be slow)
X_shap = X_test.sample(min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values computed for {len(X_shap)} samples x {X_shap.shape[1]} features")

# --- 1. Beeswarm summary plot ---
fig1, ax1 = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, plot_type="dot",
                  max_display=15, show=False)
plt.title("SHAP Feature Impact (Beeswarm) - Churn Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_beeswarm.png")

# --- 2. Bar chart - mean |SHAP| per feature ---
mean_shap = pd.Series(
    abs(shap_values).mean(axis=0),
    index=X_shap.columns
).sort_values(ascending=False).head(15)

fig2, ax2 = plt.subplots(figsize=(10, 6))
mean_shap.plot(kind="barh", ax=ax2, color="steelblue")
ax2.invert_yaxis()
ax2.set_xlabel("Mean |SHAP Value| (average impact on model output)")
ax2.set_title("Top-15 Features by SHAP Importance - Churn Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_feature_importance.png")

# --- 3. Waterfall plot for a single high-risk customer ---
# Pick the customer most likely to churn
high_risk_idx = y_prob_tuned.argmax()
shap_single = explainer(X_shap.iloc[[high_risk_idx]])

fig3, ax3 = plt.subplots(figsize=(10, 6))
shap.plots.waterfall(shap_single[0], max_display=12, show=False)
plt.title(f"SHAP Waterfall - Highest Churn Risk Customer (prob={y_prob_tuned[high_risk_idx]:.2%})",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_waterfall_highrisk.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_waterfall_highrisk.png")
print("\nKey SHAP Insights:")
for feat, val in mean_shap.head(5).items():
    print(f"  {feat}: {val:.4f} avg |SHAP|")


### SHAP Explainability
SHAP (SHapley Additive exPlanations) quantifies each feature's contribution to individual predictions — essential for business stakeholder communication.

In [ ]:
# SHAP Values - Explainability for Business Stakeholders
import shap
import matplotlib.pyplot as plt
import pandas as pd

print("Computing SHAP values for the tuned XGBoost model...")

# TreeExplainer is the fastest and most accurate for XGBoost
explainer = shap.TreeExplainer(best_xgb)

# Use a sample for speed (SHAP on full test set can be slow)
X_shap = X_test.sample(min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values computed for {len(X_shap)} samples x {X_shap.shape[1]} features")

# --- 1. Beeswarm summary plot ---
fig1, ax1 = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, plot_type="dot",
                  max_display=15, show=False)
plt.title("SHAP Feature Impact (Beeswarm) - Churn Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_beeswarm.png")

# --- 2. Bar chart - mean |SHAP| per feature ---
mean_shap = pd.Series(
    abs(shap_values).mean(axis=0),
    index=X_shap.columns
).sort_values(ascending=False).head(15)

fig2, ax2 = plt.subplots(figsize=(10, 6))
mean_shap.plot(kind="barh", ax=ax2, color="steelblue")
ax2.invert_yaxis()
ax2.set_xlabel("Mean |SHAP Value| (average impact on model output)")
ax2.set_title("Top-15 Features by SHAP Importance - Churn Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_feature_importance.png")

# --- 3. Waterfall plot for a single high-risk customer ---
# Pick the customer most likely to churn
high_risk_idx = y_prob_tuned.argmax()
shap_single = explainer(X_shap.iloc[[high_risk_idx]])

fig3, ax3 = plt.subplots(figsize=(10, 6))
shap.plots.waterfall(shap_single[0], max_display=12, show=False)
plt.title(f"SHAP Waterfall - Highest Churn Risk Customer (prob={y_prob_tuned[high_risk_idx]:.2%})",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/shap_waterfall_highrisk.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: reports/shap_waterfall_highrisk.png")
print("\nKey SHAP Insights:")
for feat, val in mean_shap.head(5).items():
    print(f"  {feat}: {val:.4f} avg |SHAP|")
